# 3장 · 분석 전에 자료를 안다

이 노트북은 구글 **Colab**에서 바로 실행됩니다. 설치는 없고, 구글 계정만 있으면 됩니다.

**읽는 법.** 지금 읽고 있는 이런 글은 설명이라 실행할 것이 없습니다. 코드는 **왼쪽에 `[ ]` 와 ▶ 단추가 붙은 상자**이고, 그 ▶ 또는 `Shift`+`Enter` 로 실행합니다.

**순서.** 번호 차례대로 끝까지 갑니다. **1 준비**부터 시작해 마지막 번호까지 위에서 아래로 내려가면 됩니다. ⚠ 가운데부터 누르면 앞에서 만든 것이 없어 오류가 납니다.

📖 본문 학습 페이지: [3장 · 분석 전에 자료를 안다](https://grow.minds.kr/textbooks/css-methods/causal/book/ch03-자료-다루기.html)

## 1. 준비

아래 상자 **둘**을 차례로 실행하세요. 둘 다 해야 그다음이 돌아갑니다.

1. 첫째 = 자료와 코드를 내려받습니다. 몇 초 걸리고 「경고」 문구가 떠도 정상입니다. 상자 아래에 `/content/css-methods-causal-code` 가 찍히면 성공입니다.
2. 둘째 = 도구를 불러옵니다. **준비 끝**이라는 줄이 찍히면 됩니다.

⚠ 한참 뒤에 돌아와 `pd`·`exp`가 없다거나 파일이 없다는 오류가 나면 코랩이 연결을 끊은 것입니다. 자료는 내 컴퓨터가 아니라 코랩의 컴퓨터에 있었으므로 함께 사라졌습니다. **이 두 상자를 위에서부터 다시** 누르면 됩니다(첫째 상자는 여러 번 눌러도 됩니다).

> **어느 것이 상자인가.** 왼쪽에 `[ ]` 표시와 **▶** 단추가 붙은 것이 코드 상자입니다. 글씨에 색이 섞여 있습니다. 지금 읽고 있는 이런 글에는 `[ ]` 도 ▶ 도 없습니다.

In [ ]:
# 이 책의 데이터·코드를 코랩으로 내려받습니다(수 초). 다시 눌러도 됩니다.
%cd /content
![ -d css-methods-causal-code ] || git clone -q https://github.com/dataminds/css-methods-causal-code.git
%cd /content/css-methods-causal-code

In [ ]:
import pandas as pd, numpy as np
from scipy import stats

def load(name, clean=True):
    df = pd.read_csv(f"data/journey_{name}.csv")
    return df[df.attn_1 == 1] if clean and "attn_1" in df else df

def ols(y, X):                      # 절편 포함 최소제곱 → (계수, 표준오차, p, R^2)
    y = np.asarray(y, float)
    X1 = np.column_stack([np.ones(len(y))] + [np.asarray(x, float) for x in X])
    b, *_ = np.linalg.lstsq(X1, y, rcond=None)
    resid = y - X1 @ b
    n, k = X1.shape
    se = np.sqrt(np.diag(resid @ resid / (n - k) * np.linalg.inv(X1.T @ X1)))
    p = 2 * stats.t.sf(np.abs(b / se), n - k)
    r2 = 1 - (resid @ resid) / ((y - y.mean()) @ (y - y.mean()))
    return b, se, p, r2

def cohen_d(a, b):
    sp = np.sqrt(((len(a)-1)*a.std(ddof=1)**2 + (len(b)-1)*b.std(ddof=1)**2) / (len(a)+len(b)-2))
    return (a.mean() - b.mean()) / sp

def cronbach(items):
    items = np.asarray(items, float); k = items.shape[1]
    return k/(k-1) * (1 - items.var(axis=0, ddof=1).sum() / items.sum(axis=1).var(ddof=1))

print("준비 끝. 데이터와 도우미 함수를 불러왔습니다.")


💡 **방금 둘째 상자에 코드가 길게 펼쳐졌는데, 지금 이해하지 않아도 됩니다.**

이 책 전체가 쓰는 **도우미 넷**을 미리 만들어 두는 곳이라 그렇습니다. 지금은 이름과 하는 일만 훑고 넘어가세요. 안에 든 식은 필요한 장에서 하나씩 만납니다.

| 이름 | 하는 일 | 처음 쓰는 곳 |
|---|---|---|
| `load` | 자료 한 벌을 불러오고 주의 점검 실패자를 걸러 낸다 | 2장 |
| `cronbach` | 척도 신뢰도(알파) | 4장 |
| `cohen_d` | 두 집단 차이의 효과크기 | 13장 |
| `ols` | 회귀의 계수·표준오차·p·R² | 14장 |

⚠ `np.linalg.lstsq` 처럼 낯선 이름이 보여도 괜찮습니다. **그것이 이 책이 「계산은 AI가」라고 말하는 곳입니다.**

⭐ **그리고 이 넷은 여러분이 검증할 대상이 아닙니다.** 이 책이 검증해 둔 도구입니다(배포 때마다 대조 배터리가 이 함수들의 출력을 본문 수치와 맞춰 봅니다). 여러분이 읽어야 할 코드는 **AI가 준 코드**이고, 그 훈련은 3장부터 시작합니다.

📖 이 넷을 다시 보고 싶으면 책 부록 E.1b 에 같은 코드가 있습니다.

## 2. 정제: 몇 명이 남나
주의 점검 통과자만 남깁니다. 380→366, 590→566.

In [ ]:
exp = load("exp"); svy = load("svy")
print("실험판:", 380, "->", len(exp), " 설문판:", 590, "->", len(svy))

## 3. 열의 저장 형식 (§3.2)
열마다 저장 형식을 봅니다. 번호 · 성별 · 문항은 정수형, 척도 점수는 실수형입니다. 저장 형식은 척도 수준(5장 §5.4)과 다릅니다. 성별은 정수형으로 저장돼 있어도 이름표입니다.

In [ ]:
print(svy[["id", "gender", "mil_1", "mil"]].dtypes.astype(str).tolist())
print(svy.dtypes.astype(str).value_counts().to_dict())
# ['int64', 'int64', 'int64', 'float64']
# {'float64': 26, 'int64': 25}

## 4. 「모름」이 섞인 열 (§3.2)
나이 칸에 「모름」이 하나 섞이면 열 전체가 글자로 읽힙니다. 합을 내면 수를 더하지 않고 글자를 이어 붙이고, 숫자로 바꾸면 그 칸은 오류 메시지 없이 빈칸이 됩니다. 바꾼 뒤 빈칸 수(1)를 함께 봅니다.

In [ ]:
import io
t = pd.read_csv(io.StringIO("id,age\n1,23\n2,모름\n3,41\n"))     # 나이 칸에 「모름」 하나
print(pd.api.types.is_numeric_dtype(t["age"]), t["age"].sum())  # 숫자형인가 · 합
age = pd.to_numeric(t["age"], errors="coerce")                  # 숫자로 못 바꾸는 칸은 빈칸으로
print(age.tolist(), age.isna().sum(), age.mean())
print(pd.Series([5, 7]).dtype, pd.Series([5, None]).dtype)       # 빈칸이 생기면 정수형 → 실수형
# False 23모름41
# [23.0, nan, 41.0] 1 32.0
# int64 float64

## 5. 파생변수 검산
21문항 평균을 직접 만들어, 저장된 hjs와 전원 일치하는지 두 경로로 확인.

In [ ]:
items = [c for c in svy.columns if c.startswith("hjs_") and c != "hjs"]
print("1번 응답자 합÷21 =", round(svy[items].iloc[0].sum()/21, 3))   # 4.571
print("내가 만든 열과 저장된 열의 최대 차이 =", round((svy[items].mean(axis=1)-svy.hjs).abs().max(), 4))

## 6. 기준을 하나씩 얹어 보기 (§3.3 상자)
정제 기준 넷을 차례로 얹어 남는 인원과 여정–의미 상관을 봅니다. 인원은 590 → 517로 줄지만 상관은 거의 안 움직입니다.

In [ ]:
raw = load("svy", clean=False)
items_r = [c for c in raw.columns if c.startswith("hjs_")]
sd_item = raw[items_r].std(axis=1)                 # 문항 사이 흩어짐 (0 에 가까우면 일자 응답)
rules = {"정제 없음":   raw.attn_1 > -1,
         "주의점검":     raw.attn_1 == 1,
         "+ 일자응답":   (raw.attn_1 == 1) & (sd_item >= 0.5),
         "+ 나이 19~65": (raw.attn_1 == 1) & (sd_item >= 0.5) & raw.age.between(19, 65)}
for lab, m in rules.items():
    sub = raw[m]
    print(lab, len(sub), round(sub.hjs.corr(sub.mil), 3))
# 정제 없음 590 0.635 / 주의점검 566 0.629 / + 일자응답 563 0.624 / + 나이 19~65 517 0.628

## 7. 표본이 작으면 기준 선택이 무거워진다 (실습 4)
566명과 60명을 300번씩 뽑아, 네 기준이 내는 상관의 최대 격차를 봅니다. 먼저 60명이면 격차가 얼마나 커질지 예측해 적고 돌리세요.

In [ ]:
rng = np.random.default_rng(73)
for n in (566, 60):
    격차 = []
    for _ in range(300):
        i = rng.choice(len(raw), n, replace=False)      # n명을 무작위로 뽑는다
        sub, sv = raw.iloc[i], sd_item.iloc[i]
        ms = [sub.attn_1 > -1, sub.attn_1 == 1, (sub.attn_1 == 1) & (sv >= 0.5),
              (sub.attn_1 == 1) & (sv >= 0.5) & sub.age.between(19, 65)]
        rs = [sub[m].hjs.corr(sub[m].mil) for m in ms]
        격차.append(max(rs) - min(rs))                   # 네 상관의 최대 격차
    print(n, round(np.mean(격차), 3), round(max(격차), 3))
# 566 0.011 0.014
# 60 0.028 0.111

## 8. long과 wide, 그리고 결측
패널판을 wide로 펼치면 이탈이 결측으로 드러납니다(2차 80, 3차 150).

In [ ]:
panel = pd.read_csv("data/journey_panel.csv")
w = panel.pivot(index="id", columns="wave", values="mil")
print("2차 결측:", int(w[2].isna().sum()), " 3차 결측:", int(w[3].isna().sum()))

## 9. 빈칸의 기본 동작 (§3.6)
3차 삶의 의미에는 빈칸이 150칸 있습니다. pandas 평균은 빈칸을 빼고 350명으로 내고(4.926), numpy 평균은 nan을 냅니다. 빈칸은 `==`로는 찾지 못하고 `isna()`로 셉니다.

In [ ]:
m3 = w[3]                                            # 3차 삶의 의미 (빈칸 150칸)
print(round(m3.mean(), 3), m3.to_numpy().mean())     # pandas 는 빈칸을 빼고, numpy 는 nan
print(m3.isna().sum(), (m3 == float("nan")).sum())   # 빈칸 세기 = isna 는 150, == 는 0
# 4.926 nan
# 150 0

## 10. 값이 아닌 값과 범주의 순서 (§3.6 · 부록 B §B.4)
`None`은 숫자 열에서 빈칸(NaN)이 되고, 0으로 나눈 칸은 무한대(inf)가 돼 평균까지 무한대로 만듭니다. 순서 있는 범주는 순서를 알려 주지 않으면 글자순으로 늘어섭니다.

In [ ]:
x = pd.Series([4, None, 6])                                       # None 은 숫자 열에서 NaN 이 된다
print(x.tolist(), x.mean(), x.count())
ratio = pd.Series([3.0, 5.0, 4.0]) / pd.Series([1.0, 0.0, 2.0])   # 둘째 사람은 0으로 나눈다
print(ratio.tolist(), ratio.mean(), int(np.isinf(ratio).sum()))
edu = pd.Series(["중졸", "대졸", "고졸", "대졸", "고졸"])
print(edu.value_counts().sort_index().index.tolist())            # 순서를 알려 주지 않으면 글자순
edu = pd.Series(pd.Categorical(edu, categories=["중졸", "고졸", "대졸"], ordered=True))
print(edu.value_counts().sort_index().index.tolist(), int((edu >= "고졸").sum()))
# [4.0, nan, 6.0] 5.0 2
# [3.0, inf, 2.0] inf 1
# ['고졸', '대졸', '중졸']
# ['중졸', '고졸', '대졸'] 4

## 11. 직접 바꿔 보기
위 셀의 숫자(씨앗 73, 표본 크기, 제외 기준 등)를 바꿔 다시 실행해 보세요. 결과가 어떻게 달라지나요?

> **검증 로그(부록 B)**: 무엇을 바꿨고, 무엇이 나왔고, 예상과 같았는지 한 문단으로 적어 두세요. 실행이 아니라 검증이 이 책의 핵심입니다.